In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from LinearRegressionClass import LinearRegressionGD  #线性回归模型类
from LogisticRegressionClass import LogisticRegressionGD  #逻辑回归模型类

PATH = "D:\QG训练营\winequality-red.csv"  #红酒质量数据集
Data_Wine_Quality = pd.read_csv(PATH , sep=';')
print("Data_Wine_Quality即红酒数据如下：\n")
print(Data_Wine_Quality)

X = Data_Wine_Quality.drop('quality', axis=1).to_numpy()  #X：所有特征
y_reg = Data_Wine_Quality['quality'].values  #回归任务 y_reg：所有真实值
y_binary = (Data_Wine_Quality['quality'] > 6).astype(int).values  #分类任务 y_binary：逻辑值（0即小于等于6，1即大于6）
print(X)
print(y_reg)
print(y_binary)

#函数：划分训练集train和测试集test
def train_test_split(X, y, test_size=0.2, random_state=2026):
    np.random.seed(random_state)
    n_samples = len(X)
    indices = np.random.permutation(n_samples)  #获取打乱的索引
    test_size = int(n_samples * test_size)  #测试集大小（n_samples的0.2倍）
    test_indices = indices[:test_size]  #测试集索引
    train_indices = indices[test_size:]  #训练集索引
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]  #依次返回训练集特征、训练集特征、训练集真实值、测试集真实值


#函数：z分数归一化
def standardize_manual(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1  #防止除以0
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    return X_train_scaled, X_test_scaled





#==============================<主 流 程>==============================

X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, random_state=2025)  #回归任务
_, _, y_train_bin, y_test_bin = train_test_split(X, y_binary)  #分类任务
print("\n")
print(f"训练集：{X_train}")
print(f"训练集真实值：{y_train_reg}")

X_train, X_test = standardize_manual(X_train, X_test)
print(f"训练集样本数量: {X_train.shape[0]}, 测试集样本数量: {X_test.shape[0]}")
print(f"训练集特征数量：{X_train.shape[1]}，测试集特征数量：{X_test.shape[1]}")

#线性回归模型：
print('='*50)
print("线性回归模型训练：")
model_reg = LinearRegressionGD()
model_reg.fit(X_train, y_train_reg)  #开始训练

print('='*50)
print("比较训练集与测试集情况：")

lin_train_loss = model_reg.evaluate(X_train, y_train_reg)["mse"]  #训练集训练完毕的均方误差
print(f"训练集均方误差：{lin_train_loss}")

lin_test_loss = model_reg.evaluate(X_test, y_test_reg)["mse"]  #测试集训练完毕的均方误差
print(f"测试集均方误差：{lin_test_loss}")

if abs(lin_train_loss - lin_test_loss) <= 0.1:
    print("拟合良好，训练集与测试集的均方误差之差<=0.1")
    print("注意过拟合")
elif abs(lin_train_loss - lin_test_loss) > 0.1:
    print("欠拟合，训练集与测试集的均方误差之差>0.1")

#线性回归模型loss曲线
plt.figure(figsize=(10, 5))
plt.plot(model_reg.losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Linear Regression - Training Loss Curve')
plt.axhline(y=0.4169, color='r', linestyle='--', label='Final Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


#逻辑回归模型
print('='*50)
print("逻辑回归模型训练：")
model_bin = LogisticRegressionGD()
model_bin.fit(X_train, y_train_bin) #开始训练

print('='*50)
print("比较训练集与测试集情况：")
logi_train_f1 = model_bin.evaluate(X_train, y_train_bin)["f1"]
logi_train_accuracy = model_bin.evaluate(X_train, y_train_bin)["accuracy"]
print(f"训练集的f1分数：{logi_train_f1}")
print(f"训练集的准确率：{logi_train_accuracy}")

logi_test_f1 = model_bin.evaluate(X_test, y_test_bin)["f1"]
logi_test_accuracy = model_bin.evaluate(X_test, y_test_bin)["accuracy"]
print(f"测试集的f1分数：{logi_test_f1}")
print(f"测试集的准确率：{logi_test_accuracy}")

if abs(logi_train_accuracy - logi_test_accuracy) <= 0.1:
    print("训练集和测试集准确度相近，误差小于等于0.1")
else:
    print("训练集和测试集准确率误差大于0.1")

#逻辑回归loss曲线
plt.subplot(1, 2, 2)
plt.plot(model_bin.losses)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Logistic Regression - Loss Curve')
plt.tight_layout()
plt.show()

